In [ ]:
# Cell 1: Imports & Data Loading
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load Master Profile (Population Data)
df = pd.read_csv('puerto_rico_master_profile_2010_2024.csv')

# We focus on the latest data (2024) to assess CURRENT vulnerability
df_2024 = df[df['year'] == 2024].copy()
df_2024['municipio_clean'] = df_2024['municipio'].str.replace(' Municipio', '').str.strip()

print(f"Loaded 2024 profile for {len(df_2024)} municipalities.")
df_2024.head()

Dataset loaded with 1014 records across 13 years.


In [ ]:
# Cell 2: Feature Engineering (Defining "Vulnerability")
# Your proposal defines vulnerability via Age, Poverty, and Shocks.

# 1. "Shock" Variable: Calculate Population Drop % since 2010
df_2010 = df[df['year'] == 2010][['municipio', 'total_population']].rename(columns={'total_population': 'pop_2010'})
df_2024 = pd.merge(df_2024, df_2010, on='municipio')

# A negative number means a drop (e.g., -15%)
df_2024['pop_change_pct'] = ((df_2024['total_population'] - df_2024['pop_2010']) / df_2024['pop_2010']) * 100

# 2. Select Features for Clustering
# - median_income_real: Economic Resilience
# - unemployment_rate_pct: Economic Stress
# - pop_change_pct: Demographic Shock
# - hs_graduate_pct: Educational Capital
features = ['median_income_real', 'unemployment_rate_pct', 'pop_change_pct', 'hs_graduate_pct']

# 3. Standardize the Data (Required for K-Means)
X = df_2024[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features Standardized. Ready for Clustering.")

In [ ]:
# Cell 3: The Elbow Method (Optimization)
# This calculates the "Within-Cluster Sum of Squares" (WCSS) to find the optimal K
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), wcss, marker='o', color='crimson')
plt.title('The Elbow Method (finding the optimal K)', fontsize=16)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.grid(True)
plt.show()

In [ ]:
# Cell 4: Applying K-Means (The Vulnerability Score)
# Based on the Elbow plot, K=3 or K=4 is usually best. 
# Your proposal suggests "3-4 Risk Profiles", so let's use 3 (Low, Med, High).

kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
df_2024['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# Analyze the clusters to name them (e.g., "High Risk" vs "Resilient")
cluster_summary = df_2024.groupby('Cluster_ID')[features].mean()
display(cluster_summary)

# Visualizing the Separation
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_2024, 
    x='median_income_real', 
    y='pop_change_pct', 
    hue='Cluster_ID', 
    palette='viridis', 
    s=150, 
    edgecolor='black'
)
plt.title('Vulnerability Clusters: Income vs. Population Decline', fontsize=15)
plt.xlabel('Real Median Income ($)')
plt.ylabel('Population Change since 2010 (%)')
plt.axhline(0, color='gray', linestyle='--')
plt.show()